### DASHBOARD DE ACOMPANHAMENTO DE INVESTIMENTOS

Objetivo: automatizar a visualização de investimentos a partir de índices atualizados.

---

Ideias principais:

Criar um dashboard em PowerBI para visualização dos investimentos
- Porporção dos tipos de investimentos;
- Variação da cotação de cada investimento;
- DY;
- Histórico de variação dos preços;
- Renda fixa/Renda variável.

Back-end com API's Python.

---

Questões:
Como inserir novos dados? Tabela de compra/venda como input para o Python --> usa csv antigo e o novo, e cria um novo.

---

Adicional: implementar resumo dos investimentos com LLM.

---
### Podemos usar um SQL (SQLite ou PostgreSQL) para armazenar os dados de input e output. Talve
Input:
- ticker da ação;
- quantidade de cotas;
- valor comprado (ou quantidade de cotas);
- taxas de compra/venda;
- taxas de adm;

In [4]:
%pip install python-dotenv requests

  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.


In [10]:
# import libraries
import pandas as pd
import numpy as np
import os
import requests
from dotenv import load_dotenv


In [29]:
load_dotenv()

BASE_URL = "https://brapi.dev/api/v2"
TOKEN = os.getenv("BRAPI_TOKEN")

def get_stock_quote(symbols):
    url = f"{BASE_URL}/stocks/quote"

    params = {
        "symbols": ",".join(symbols),
        "token": TOKEN
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    # Pega o JSON da resposta
    json_response = response.json()
    resultados = json_response.get("results", [])

    # EXTRAÇÃO DO DICIONÁRIO 'data':
    # Percorre cada ação encontrada e extrai apenas a parte que importa (a chave 'data')
    dados_limpos = [acao['data'] for acao in resultados if 'data' in acao]
    
    return dados_limpos

# Chamando a função
dados_acoes = get_stock_quote(["PETR4", "VALE3", "ITUB4"])

# Cria o DataFrame passando a lista de dicionários extraídos
df = pd.DataFrame(dados_acoes)

# visualização de algumas colunas
df_resumo = df[['shortName', 'regularMarketPrice', 'regularMarketDayHigh', 'regularMarketDayLow','regularMarketChangePercent','logourl']]
df_resumo

,shortName,regularMarketPrice,regularMarketDayHigh,regularMarketDayLow,regularMarketChangePercent,logourl
0,PETR4,42.47,42.76,41.72,0.90,https://icons.brapi.dev/icons/PETR4.svg
1,VALE3,71.41,71.80,70.69,0.15,https://icons.brapi.dev/icons/VALE3.svg
2,ITUB4,38.38,38.98,38.17,-1.59,https://icons.brapi.dev/icons/ITUB4.svg


### Principais dados a serem exportados da API
- Symbol (shortname) que representa o ticker da ação, por exemplo: ITUB4
- regularMarketPrice que mostra o preço atual da ação, permitindo calcular o valor atualizado da carteira
- regularMarketChangePercent indicando a variação percentual do dia. Podemos ainda adicionar o preço máximo e mínimo da ação no dia (high and low)
- logourl isso será importante no PowerBI, mostra o URL da logo da empresa.

Um dos pontos que a API Brapi não fornece é o dividendo anual (DY - dividend yield)